In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Labs"
)

xpt_file = DATA_DIR / "TRIGLY_F.xpt"
csv_file = DATA_DIR / "TRIGLY_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched source values.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 TRIGLY_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTSAF2YR",
    "LBXTR",
    "LBDTRSI",
    "LBDLDL",
    "LBDLDLSI"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "TRIGLY_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 3581:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 6:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 3,581 rows x 6 columns."
)


# ============================================================
# 4. CHECK FOR KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# Do NOT globally replace this value before validating
# which variable contains it.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Validated against:
#
# 1. Attached TRIGLY_F.xpt
# 2. CDC 2009-2010 TRIGLY_F codebook
#
# CDC WTSAF2YR valid range begins at 0.
#
# The attached XPT contains exactly:
#
#     WTSAF2YR = 396 tiny-zero values
#

EXPECTED_TINY_COUNTS = {
    "WTSAF2YR": 396
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated TRIGLY_F pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 396:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total}"
    )


print(
    "\nPASS: Tiny-value pattern matches "
    "validated TRIGLY_F pattern."
)


# ============================================================
# 6. RESTORE VALIDATED WTSAF2YR ZEROS
# ============================================================

#
# Restore ONLY the validated WTSAF2YR tiny values.
#
# Do NOT globally replace tiny values in unrelated fields.
#

tiny_mask = (
    df["WTSAF2YR"]
    .eq(TINY_VALUE)
)


correction_count = int(
    tiny_mask.sum()
)


if correction_count != 396:

    raise ValueError(
        "Unexpected WTSAF2YR tiny-value count."
    )


df.loc[
    tiny_mask,
    "WTSAF2YR"
] = 0


print("\n--- ZERO RESTORATION ---")


print(
    "WTSAF2YR values restored to zero:",
    correction_count
)


if int(
    df["WTSAF2YR"]
    .eq(0)
    .sum()
) != 396:

    raise ValueError(
        "WTSAF2YR zero restoration failed."
    )


print(
    "PASS: 396 WTSAF2YR zeros restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the final corrected
# dataset SHOULD contain.
#
# The only expected numeric difference from source_df is:
#
#     WTSAF2YR tiny -> 0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["WTSAF2YR"].eq(TINY_VALUE),
    "WTSAF2YR"
] = 0


# ============================================================
# 8. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# NEVER round legitimate decimal measurements.
#
# This applies to:
#
# - Laboratory measurements
# - Sample weights
# - Dietary values
# - Examination measurements
#
# Identify genuine fractional variables before
# converting anything to Int64.
#

print("\n--- GENUINE DECIMAL CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "genuine fractional values"
        )


# ============================================================
# 9. VALIDATE EXPECTED DECIMAL VARIABLES
# ============================================================

#
# Genuine decimal variables:
#
# WTSAF2YR:
#     Fasting subsample weight
#
# LBDTRSI:
#     Triglyceride mmol/L
#
# LBDLDLSI:
#     LDL cholesterol mmol/L
#

DECIMAL_COLUMNS = [
    "WTSAF2YR",
    "LBDTRSI",
    "LBDLDLSI"
]


if fractional_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern differs "
        "from validated TRIGLY_F pattern.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "\nPASS: Genuine decimal variables identified:"
)


for col in DECIMAL_COLUMNS:

    print(
        "   ",
        col
    )


print(
    "\nPASS: Decimal variables will NOT be "
    "rounded or converted to Int64."
)


# ============================================================
# 10. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51641
    or
    df["SEQN"].max()
    != 62158
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 11. FORMAT ONLY WHOLE-NUMBER VARIABLES
# ============================================================

#
# Whole-number variables:
#
#     SEQN
#     LBXTR
#     LBDLDL
#
# Genuine decimal variables remain float.
#
# NO .round() is used.
#

INTEGER_COLUMNS = [
    "SEQN",
    "LBXTR",
    "LBDLDL"
]


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN, LBXTR, and LBDLDL formatted "
    "without unnecessary .0."
)


print(
    "PASS: Genuine decimal variables remain unchanged."
)


# ============================================================
# 12. CDC CHECK - WTSAF2YR
# ============================================================

#
# CDC:
#
# Fasting Subsample 2 Year MEC Weight
#
# Range:
#     0 to 337218.92
#
# Non-missing:
#     3,581
#
# Missing:
#     0
#
# The 396 tiny XPORT values represent zero.
#

s = df["WTSAF2YR"]


print("\n--- WTSAF2YR ---")


print(
    "Minimum:",
    s.min()
)


print(
    "Maximum:",
    s.max()
)


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    np.isclose(
        s.min(),
        0,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        337218.92,
        rtol=0,
        atol=0
    )
    and
    int(
        s.eq(0)
        .sum()
    )
    == 396
    and
    int(
        s.notna()
        .sum()
    )
    == 3581
    and
    int(
        s.isna()
        .sum()
    )
    == 0
):

    raise ValueError(
        "WTSAF2YR differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: WTSAF2YR matches CDC."
)


print(
    "PASS: Non-zero WTSAF2YR decimals preserved."
)


# ============================================================
# 13. CDC CHECK - LBXTR
# ============================================================

#
# CDC:
#
# Triglycerides (mg/dL)
#
# Range:
#     18 to 2742
#
# Non-missing:
#     3,357
#
# Missing:
#     224
#

s = df["LBXTR"]


print("\n--- LBXTR ---")


print(
    "Minimum:",
    s.min()
)


print(
    "Maximum:",
    s.max()
)


print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    s.min() == 18
    and
    s.max() == 2742
    and
    int(
        s.notna()
        .sum()
    )
    == 3357
    and
    int(
        s.isna()
        .sum()
    )
    == 224
):

    raise ValueError(
        "LBXTR differs from CDC."
    )


print(
    "PASS: LBXTR matches CDC."
)


# ============================================================
# 14. CDC CHECK - LBDTRSI
# ============================================================

#
# CDC:
#
# Triglycerides (mmol/L)
#
# Range:
#     0.203 to 30.957
#
# Non-missing:
#     3,357
#
# Missing:
#     224
#
# Genuine decimal variable.
#

s = df["LBDTRSI"]


print("\n--- LBDTRSI ---")


print(
    "Minimum:",
    s.min()
)


print(
    "Maximum:",
    s.max()
)


print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    np.isclose(
        s.min(),
        0.203,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        30.957,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    )
    == 3357
    and
    int(
        s.isna()
        .sum()
    )
    == 224
):

    raise ValueError(
        "LBDTRSI differs from CDC."
    )


print(
    "PASS: LBDTRSI matches CDC."
)


print(
    "PASS: LBDTRSI decimals preserved."
)


# ============================================================
# 15. CDC CHECK - LBDLDL
# ============================================================

#
# CDC:
#
# LDL Cholesterol (mg/dL)
#
# Range:
#     13 to 266
#
# Non-missing:
#     3,308
#
# Missing:
#     273
#

s = df["LBDLDL"]


print("\n--- LBDLDL ---")


print(
    "Minimum:",
    s.min()
)


print(
    "Maximum:",
    s.max()
)


print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    s.min() == 13
    and
    s.max() == 266
    and
    int(
        s.notna()
        .sum()
    )
    == 3308
    and
    int(
        s.isna()
        .sum()
    )
    == 273
):

    raise ValueError(
        "LBDLDL differs from CDC."
    )


print(
    "PASS: LBDLDL matches CDC."
)


# ============================================================
# 16. CDC CHECK - LBDLDLSI
# ============================================================

#
# CDC:
#
# LDL Cholesterol (mmol/L)
#
# Range:
#     0.336 to 6.879
#
# Non-missing:
#     3,308
#
# Missing:
#     273
#
# Genuine decimal variable.
#

s = df["LBDLDLSI"]


print("\n--- LBDLDLSI ---")


print(
    "Minimum:",
    s.min()
)


print(
    "Maximum:",
    s.max()
)


print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    np.isclose(
        s.min(),
        0.336,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        6.879,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    )
    == 3308
    and
    int(
        s.isna()
        .sum()
    )
    == 273
):

    raise ValueError(
        "LBDLDLSI differs from CDC."
    )


print(
    "PASS: LBDLDLSI matches CDC."
)


print(
    "PASS: LBDLDLSI decimals preserved."
)


# ============================================================
# 17. TRIGLYCERIDE CONVERSION CONSISTENCY CHECK
# ============================================================

#
# CDC states:
#
#     LBDTRSI = LBXTR * 0.01129
#
# Released values are represented to 3 decimal places.
#
# IMPORTANT:
#
# Use Decimal + ROUND_HALF_UP.
#
# Python's regular round() can behave differently
# on exact .5 boundaries due to banker's rounding.
#
# This check DOES NOT modify the data.
#

print("\n--- TRIGLYCERIDE CONVERSION CHECK ---")


conversion_failures = []


mask = (
    df["LBXTR"].notna()
    &
    df["LBDTRSI"].notna()
)


for idx in df.index[mask]:

    mg_dl = int(
        df.at[
            idx,
            "LBXTR"
        ]
    )


    expected_si = float(
        (
            Decimal(
                str(mg_dl)
            )
            *
            Decimal(
                "0.01129"
            )
        )
        .quantize(
            Decimal(
                "0.001"
            ),
            rounding=ROUND_HALF_UP
        )
    )


    actual_si = float(
        df.at[
            idx,
            "LBDTRSI"
        ]
    )


    if expected_si != actual_si:

        conversion_failures.append(
            (
                int(
                    df.at[
                        idx,
                        "SEQN"
                    ]
                ),
                mg_dl,
                actual_si,
                expected_si
            )
        )


        if len(
            conversion_failures
        ) >= 20:

            break


if conversion_failures:

    print(
        conversion_failures
    )

    raise ValueError(
        "LBDTRSI conversion consistency check failed."
    )


print(
    "PASS: LBDTRSI is consistent with "
    "LBXTR x 0.01129."
)


# ============================================================
# 18. LDL SI CONVERSION CONSISTENCY CHECK
# ============================================================

#
# CDC states:
#
#     LBDLDLSI = LBDLDL * 0.02586
#
# Released values are represented to 3 decimal places.
#
# Again use ROUND_HALF_UP only for validation.
#
# The original LBDLDLSI values are NOT changed.
#

print("\n--- LDL SI CONVERSION CHECK ---")


conversion_failures = []


mask = (
    df["LBDLDL"].notna()
    &
    df["LBDLDLSI"].notna()
)


for idx in df.index[mask]:

    mg_dl = int(
        df.at[
            idx,
            "LBDLDL"
        ]
    )


    expected_si = float(
        (
            Decimal(
                str(mg_dl)
            )
            *
            Decimal(
                "0.02586"
            )
        )
        .quantize(
            Decimal(
                "0.001"
            ),
            rounding=ROUND_HALF_UP
        )
    )


    actual_si = float(
        df.at[
            idx,
            "LBDLDLSI"
        ]
    )


    if expected_si != actual_si:

        conversion_failures.append(
            (
                int(
                    df.at[
                        idx,
                        "SEQN"
                    ]
                ),
                mg_dl,
                actual_si,
                expected_si
            )
        )


        if len(
            conversion_failures
        ) >= 20:

            break


if conversion_failures:

    print(
        conversion_failures
    )

    raise ValueError(
        "LBDLDLSI conversion consistency check failed."
    )


print(
    "PASS: LBDLDLSI is consistent with "
    "LBDLDL x 0.02586."
)


# ============================================================
# 19. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected numeric change from original XPT:
#
# WTSAF2YR:
#
#     396 tiny XPORT values -> numeric 0
#
# NOTHING ELSE may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col in DECIMAL_COLUMNS:

        current_values = (
            df[col]
            .to_numpy(
                dtype=float
            )
        )

    else:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 6 variables match expected "
    "corrected XPT values exactly."
)


# ============================================================
# 20. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO float_format.
# NO rounding.
#
# Whole-number fields:
#
#     SEQN
#     LBXTR
#     LBDLDL
#
# Genuine decimal fields:
#
#     WTSAF2YR
#     LBDTRSI
#     LBDLDLSI
#
# remain decimal.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 21. RE-READ EXPORTED CSV
# ============================================================

#
# round_trip precision is used for exact decimal
# fidelity verification.
#

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 22. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 6 CSV columns match "
    "expected corrected XPT values exactly."
)


# ============================================================
# 23. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 24. CHECK CSV FOR REMAINING TINY VALUES
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 25. WHOLE-NUMBER .0 FORMAT CHECK
# ============================================================

#
# IMPORTANT:
#
# DO NOT search the entire CSV for ".0".
#
# Genuine decimal fields may legitimately contain:
#
#     0.0
#     1.0
#     3.051
#
# We check ONLY:
#
#     SEQN
#     LBXTR
#     LBDLDL
#

print("\n--- WHOLE-NUMBER FORMAT CHECK ---")


dot_zero_errors = []


with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(
        f
    )


    for row_number, row in enumerate(
        reader,
        start=2
    ):

        for col in INTEGER_COLUMNS:

            value = row[col]


            if (
                value != ""
                and
                value.endswith(".0")
            ):

                dot_zero_errors.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


                if len(
                    dot_zero_errors
                ) >= 20:

                    break


        if len(
            dot_zero_errors
        ) >= 20:

            break


print(
    "Unnecessary '.0' errors:",
    len(
        dot_zero_errors
    )
)


if dot_zero_errors:

    print(
        dot_zero_errors
    )

    raise ValueError(
        "Unexpected .0 formatting remains "
        "in whole-number fields."
    )


print(
    "PASS: SEQN, LBXTR, and LBDLDL "
    "have no unnecessary .0."
)


# ============================================================
# 26. EXACT DECIMAL FIDELITY CHECK
# ============================================================

#
# Explicitly verify all genuine decimal variables.
#
# These must survive:
#
# XPT
# -> validated zero correction
# -> CSV
#
# with zero unexpected numeric change.
#

print("\n--- DECIMAL VALUE FIDELITY ---")


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Decimal values changed in:\n"
        + str(decimal_failures)
    )


print(
    "PASS: All genuine decimal variables "
    "preserved exactly."
)


# ============================================================
# 27. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print("\n--- SCIENTIFIC NOTATION CHECK ---")


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 28. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51641
    or
    csv_df["SEQN"].max()
    != 62158
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 29. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "TRIGLY_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "WTSAF2YR zero restorations:",
    correction_count
)


print(
    "Whole-number fields:",
    ", ".join(
        INTEGER_COLUMNS
    )
)


print(
    "Decimal variables preserved:",
    ", ".join(
        DECIMAL_COLUMNS
    )
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0 in whole-number fields: 0 expected"
)


print(
    "Lab decimal values changed: 0 expected"
)

NHANES 2009-2010 TRIGLY_F XPT -> CSV
Rows: 3,581
Columns: 6
PASS: Structure = 3,581 rows x 6 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 396

Tiny values by variable:
WTSAF2YR    396

PASS: Tiny-value pattern matches validated TRIGLY_F pattern.

--- ZERO RESTORATION ---
WTSAF2YR values restored to zero: 396
PASS: 396 WTSAF2YR zeros restored.

--- GENUINE DECIMAL CHECK ---
WTSAF2YR: 3,181 genuine fractional values
LBDTRSI: 3,357 genuine fractional values
LBDLDLSI: 3,273 genuine fractional values

PASS: Genuine decimal variables identified:
    WTSAF2YR
    LBDTRSI
    LBDLDLSI

PASS: Decimal variables will NOT be rounded or converted to Int64.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51641
Maximum: 62158
PASS: SEQN validated.

PASS: SEQN, LBXTR, and LBDLDL formatted without unnecessary .0.
PASS: Genuine decimal variables remain unchanged.

--- WTSAF2YR ---
Minimum: 0.0
Maximum: 337218.92
Zero: 396
Non-missing: 3581
Missing: 0
PASS: W